# 03 — Employment
## Trade Winds & Headwinds: USVI, 2000–2025

Sections:
1. FRED Leisure & Hospitality employment — monthly SA trend
2. FRED monthly NSA — seasonality
3. FRED annual averages
4. USVIBER tourism-related employment (2010–2022)
5. Census employment snapshots (NAICS 72, 721, 722)
6. Cross-source comparison

**Units note:** FRED values are in thousands of persons. Census employment is a point-in-time count (pay period including March 12). Do not sum FRED monthly values to get annual employment.

In [1]:
import sys, importlib
from pathlib import Path
import os

PROJ_ROOT = Path("..").resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))
os.chdir(PROJ_ROOT)

import numpy as np
import pandas as pd
import plotly.graph_objects as go


import src.cleaning as _c
import src.charts as _ch

importlib.reload(_c)
importlib.reload(_ch)


# override default notebook template, set background colors to white
# charts.py has a transparent background
import plotly.io as pio

pio.templates["notebook"] = go.layout.Template(
    layout=dict(
        paper_bgcolor="#ffffff",
        plot_bgcolor="#ffffff",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.22,
            xanchor="center",
            x=0.5,
            bgcolor="rgba(255,255,255,0)",
            font=dict(size=10.5),
            itemsizing="constant",
            tracegroupgap=0,
        ),
    )
)
pio.templates.default = "plotly_white+notebook"


from src.cleaning import (
    load_fred_employment,
    load_usviber_combined,
    load_census_2007,
    load_census_2012_basic,
    load_census_2017_basic,
    load_census_2022_basic,
    load_qcew,
    load_cpi_annual,
)
from src.charts import PALETTE, bar_chart, line_chart, add_shocks, source_note

pd.set_option("display.float_format", "{:,.1f}".format)
print("Ready.")

Ready.


In [2]:
fred = load_fred_employment()
tour = load_usviber_combined()
c07 = load_census_2007()
c12 = load_census_2012_basic()
c17 = load_census_2017_basic()
c22 = load_census_2022_basic()
print(f"FRED: {fred.date.min()} – {fred.date.max()}  ({len(fred)} months)")

FRED: 1990-01 – 2026-09  (441 months)


---
## 1. FRED Monthly — Seasonally Adjusted Trend

In [3]:
fred_plot = fred.copy()
fred_plot["date"] = fred_plot["date"].dt.to_timestamp()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=fred_plot["date"],
        y=fred_plot["emp_sa"],
        mode="lines",
        name="SA employment",
        line=dict(color=PALETTE["employment"], width=2),
    )
)
fig.update_layout(
    title=dict(
        text="USVI Leisure & Hospitality Employment — Seasonally Adjusted<br>"
        "<span style='font-size:13px;color:#555'>Thousands of persons. "
        "Leisure & Hospitality (NAICS 71+72) is broader than tourism-only employment.</span>",
        font=dict(size=17),
    ),
    yaxis_title="Employees (thousands)",
    height=440,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text="<span style='color:#888;font-size:11px'>Source: FRED SMS78000007000000001 (BLS)</span>",
            showarrow=False,
            xanchor="left",
        )
    ],
)
# Historical hurricane and COVID event markers
for x_str, label, color in [
    ("1995-09-01", "Marilyn", "#777777"),
    ("2017-09-01", "Irma & Maria", "#999999"),
    ("2020-03-01", "COVID-19", "#999999"),
]:
    fig.add_vline(
        x=pd.Timestamp(x_str),
        line=dict(color=color, width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color=color),
    )
fig.show()

---
## 2. FRED Monthly — NSA Seasonality

In [4]:
from src.charts import monthly_heatmap

fred_heat = fred.query("year >= 2000").copy()
fred_heat["year_int"] = fred_heat["year"].astype(int)
fred_heat["month_int"] = fred_heat["month"].astype(int)

fig = monthly_heatmap(
    fred_heat,
    year_col="year_int",
    month_col="month_int",
    value_col="emp_nsa",
    title="USVI L&H Employment (NSA) — Seasonal Pattern",
    subtitle="Thousands of persons. Darker = more employment.",
    source="FRED SMU78000007000000001 (BLS)",
    colorscale="Blues",
    height=600,
)
fig.show()

In [5]:
# Average seasonal profile by era
# Bins use right-closed intervals; 2017 gets its own bin as the hurricane shock year.
fred2 = fred.query("year >= 2000").copy()
fred2["era"] = pd.cut(
    fred2["year"],
    bins=[1999, 2016, 2017, 2019, 2021, 2025],
    labels=[
        "Pre-hurricane (2000—2016)",
        "Hurricane shock (2017)",
        "Initial recovery (2018—2019)",
        "COVID shock (2020—2021)",
        "Recent (2022—2025)",
    ],
)
seasonal = (
    fred2.dropna(subset=["emp_nsa", "era"])
    .groupby(["era", "month"])["emp_nsa"]
    .mean()
    .reset_index()
)
month_labels = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
era_colors = [
    PALETTE["total"],
    "#b22222",
    PALETTE["tourists"],
    PALETTE["cruise"],
    PALETTE["rooms"],
]

fig = go.Figure()
for era, color in zip(seasonal["era"].cat.categories, era_colors):
    sub = seasonal[seasonal["era"] == era].sort_values("month")
    if sub.empty:
        continue
    fig.add_trace(
        go.Scatter(
            x=[month_labels[m - 1] for m in sub["month"]],
            y=sub["emp_nsa"],
            mode="lines+markers",
            name=str(era),
            line=dict(color=color, width=2),
            marker=dict(size=5),
        )
    )

fig.update_layout(
    title=dict(
        text="L&H Employment — Average Seasonal Profile by Era (NSA)<br><span style='font-size:13px;color:#555'>Mean monthly NSA employment (thousands). 2017 shown separately.</span>",
        font=dict(size=17),
    ),
    yaxis_title="Average NSA employment (thousands)",
    height=440,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text="<span style='color:#888;font-size:11px'>Source: FRED SMU78000007000000001 (BLS)</span>",
            showarrow=False,
            xanchor="left",
        )
    ],
)
fig.show()

---
## 3. FRED Annual Averages

In [6]:
# Annual average SA employment (mean of monthly, not sum)
fred_ann = (
    fred.groupby("year")[["emp_sa", "emp_nsa"]]
    .mean()
    .reset_index()
    .rename(columns={"emp_sa": "sa_avg", "emp_nsa": "nsa_avg"})
)

fig = bar_chart(
    fred_ann.query("year >= 2000"),
    x="year",
    y="sa_avg",
    title="USVI L&H Employment — Annual Average (SA)",
    subtitle="Mean of 12 monthly SA observations. Not a sum.",
    y_label="Avg. annual employees (thousands)",
    color=PALETTE["employment"],
    source="FRED SMS78000007000000001 — author calculations",
    shock_x=[2017, 2020],
)
fig.show()

print("Annual average SA employment (thousands):")
print(fred_ann.query("year >= 2000").set_index("year").round(2).to_string())

Annual average SA employment (thousands):
      sa_avg  nsa_avg
year                 
2000     6.8      6.8
2001     6.9      6.9
2002     6.8      6.8
2003     6.9      6.9
2004     7.3      7.3
2005     7.1      7.1
2006     7.2      7.2
2007     7.3      7.3
2008     7.5      7.5
2009     6.9      6.9
2010     7.2      7.2
2011     7.4      7.4
2012     7.3      7.3
2013     7.2      7.2
2014     7.2      7.2
2015     7.4      7.4
2016     7.7      7.7
2017     6.9      6.9
2018     4.2      4.2
2019     5.6      5.6
2020     4.9      4.9
2021     5.8      5.8
2022     6.3      6.3
2023     6.8      6.8
2024     7.0      7.0
2025     6.5      6.5
2026     5.5      5.5


---
## 4. USVIBER Tourism-Related Employment

In [7]:
# USVIBER employment — available 2010–2022 only
tour_emp = tour[
    ["year", "emp_tourism_total", "emp_hotels", "emp_eating_drinking"]
].dropna(subset=["emp_tourism_total"])

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=tour_emp["year"],
        y=tour_emp["emp_hotels"],
        name="Hotels & lodging",
        marker_color=PALETTE["tourists"],
    )
)
fig.add_trace(
    go.Bar(
        x=tour_emp["year"],
        y=tour_emp["emp_eating_drinking"],
        name="Eating & drinking places",
        marker_color=PALETTE["cruise"],
    )
)
fig.update_layout(
    barmode="stack",
    title=dict(
        text="USVIBER Tourism-Related Employment (Selected Categories)<br>"
        '<span style="font-size:13px;color:#555">Number of jobs. Data ends 2022; 2023–2024 not available from DOL.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Number of jobs",
    yaxis_tickformat=",",
    height=440,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: USVIBER Annual Tourism Indicator 2024 [a] U.S. Virgin Islands Bureau of Labor Statistics</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()
print(tour_emp.set_index("year").to_string())

      emp_tourism_total  emp_hotels  emp_eating_drinking
year                                                    
2001            9,020.0         NaN                  NaN
2002            8,910.0         NaN                  NaN
2003            8,379.0         NaN                  NaN
2004            8,723.0         NaN                  NaN
2005            8,695.0         NaN                  NaN
2006            8,593.0         NaN                  NaN
2007            8,778.0         NaN                  NaN
2008            8,683.0         NaN                  NaN
2009            8,038.0         NaN                  NaN
2010            8,301.0     3,587.0              2,793.0
2011            8,469.0     3,549.0              2,959.0
2012            8,339.0     3,617.0              2,821.0
2013            8,136.0     3,519.0              2,784.0
2014            8,171.0     3,575.0              2,809.0
2015            8,159.0     3,629.0              2,765.0
2016            8,430.0     3,6

---
## 5. Census Employment Snapshots

In [8]:
# Build a comparable Census NAICS 72 / 721 / 722 table across vintages
frames = []
for df, vintage_year in [(c07, 2007), (c12, 2012), (c17, 2017), (c22, 2022)]:
    sub = df[df["naics"].astype(str).isin(["72", "721", "722"])][
        ["year", "naics", "naics_label", "estab", "rcptot_k", "payann_k", "emp"]
    ].copy()
    sub["vintage"] = vintage_year
    frames.append(sub)

census = pd.concat(frames, ignore_index=True)
# De-duplicate: each vintage file contains the prior period's data too;
# keep only the row where year matches the vintage for each file
# (or if a file contains two years, keep both)
census_key = census[census["naics"] == "72"].sort_values("year")

print("=== Census NAICS 72: Accommodation & Food Services ===")
print(
    census_key[["year", "estab", "emp", "payann_k", "rcptot_k"]]
    .drop_duplicates()
    .set_index("year")
    .to_string()
)

=== Census NAICS 72: Accommodation & Food Services ===
      estab     emp  payann_k  rcptot_k
year                                   
2002    313 5,639.0  92,357.0 331,008.0
2007    255 6,146.0 129,635.0 460,532.0
2012    279 6,864.0 160,542.0 539,442.0
2017    304 6,737.0 149,735.0 520,417.0
2022    293 5,140.0 136,885.0 536,134.0


In [9]:
# Plot Census employment for NAICS 72, 721, 722
# Explicitly prefer the later vintage when years overlap across vintage files:
# year 2012 appears in both vintage-2012 and vintage-2017 files; keep vintage-2012.
# year 2017 appears in both vintage-2017 and vintage-2022 files; keep vintage-2017.
census_dedup = census.sort_values(
    ["year", "vintage"]
).drop_duplicates(  # earlier vintage first
    subset=["year", "naics"], keep="first"
)  # keep earlier vintage

fig = go.Figure()
for naics_code, name, color in [
    ("72", "Accommodation & food services (NAICS 72)", PALETTE["total"]),
    ("721", "Accommodation (NAICS 721)", PALETTE["tourists"]),
    ("722", "Food services & drinking places (NAICS 722)", PALETTE["cruise"]),
]:
    sub = (
        census_dedup[census_dedup["naics"] == naics_code]
        .dropna(subset=["emp"])
        .sort_values("year")
    )
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub["emp"],
            mode="markers+lines",
            name=name,
            line=dict(color=color, width=2, dash="dot"),
            marker=dict(size=10, symbol="diamond"),
        )
    )

fig.update_layout(
    title=dict(
        text="Census Employment Snapshots — Accommodation & Food Services<br>"
        "<span style='font-size:13px;color:#555'>Point-in-time (March pay period). Quinquennial snapshots; not a continuous series.</span>",
        font=dict(size=17),
    ),
    yaxis_title="Paid employees (point-in-time)",
    yaxis_tickformat=",",
    height=440,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text="<span style='color:#888;font-size:11px'>Source: U.S. Census Economic Census 2007, 2012, 2017, 2022 — USVI territory level. 2017 reference date = March 2017 pay period.</span>",
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

In [10]:
# Payroll and receipts (nominal)
c_n72 = (
    census_dedup[census_dedup["naics"] == "72"]
    .dropna(subset=["payann_k"])
    .sort_values("year")
)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=c_n72["year"],
        y=c_n72["rcptot_k"] / 1000,
        name="Receipts ($M)",
        marker_color=PALETTE["total"],
    )
)
fig.add_trace(
    go.Bar(
        x=c_n72["year"],
        y=c_n72["payann_k"] / 1000,
        name="Annual payroll ($M)",
        marker_color=PALETTE["tourists"],
    )
)
fig.update_layout(
    barmode="group",
    title=dict(
        text="NAICS 72 — Receipts and Payroll (Nominal $M)<br>"
        '<span style="font-size:13px;color:#555">Census snapshots. No inflation adjustment.</span>',
        font=dict(size=17),
    ),
    yaxis_title="$M (nominal)",
    height=420,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: U.S. Census Economic Census 2007–2022 — USVI territory level</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
fig.show()

---
## 6. Cross-Source Comparison

Three employment series measure overlapping but distinct things:

| Series | Scope | Frequency | Units | Reference |
|---|---|---|---|---|
| FRED SA/NSA | Leisure & Hospitality (NAICS 71+72) | Monthly | Thousands | Pay-period average |
| USVIBER | Tourism-related (hotels, eating, transport, gift) | Annual | Jobs | Year-end survey |
| Census NAICS 72 | Accommodation & food services only | Quinquennial | Jobs | March pay period |

**Do not equate these series.** FRED is broadest (includes arts/recreation). Census NAICS 72 excludes transportation and retail. USVIBER's category list differs from standard NAICS.

In [11]:
# Side-by-side comparison for years with overlapping data
fred_ann = (
    fred.query("year >= 2000")
    .groupby("year")["emp_sa"]
    .mean()
    .mul(1000)  # convert to absolute count for comparison
    .reset_index()
    .rename(columns={"emp_sa": "fred_sa_avg"})
)
usviber_emp = tour[["year", "emp_tourism_total"]].dropna()
census_72 = (
    census_dedup[census_dedup["naics"] == "72"][["year", "emp"]]
    .dropna()
    .rename(columns={"emp": "census_n72"})
)

comp = (
    fred_ann.merge(usviber_emp, on="year", how="outer")
    .merge(census_72, on="year", how="outer")
    .sort_values("year")
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=comp["year"],
        y=comp["fred_sa_avg"],
        mode="lines",
        name="FRED L&H (SA annual avg)",
        line=dict(color=PALETTE["total"], width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=comp["year"],
        y=comp["emp_tourism_total"],
        mode="lines+markers",
        name="USVIBER tourism employment",
        line=dict(color=PALETTE["tourists"], width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=comp["year"],
        y=comp["census_n72"],
        mode="markers",
        name="Census NAICS 72 (March)",
        marker=dict(color=PALETTE["cruise"], size=12, symbol="diamond"),
    )
)

fig.update_layout(
    title=dict(
        text="Employment Cross-Source Comparison<br>"
        '<span style="font-size:13px;color:#555">Three different series with different scopes — do not assume equivalence</span>',
        font=dict(size=17),
    ),
    yaxis_title="Employees",
    yaxis_tickformat=",",
    height=460,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Sources: FRED, USVIBER, U.S. Census Economic Census</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()
print(comp.set_index("year").round(0).to_string())

      fred_sa_avg  emp_tourism_total  census_n72
year                                            
2000      6,800.0                NaN         NaN
2001      6,917.0            9,020.0         NaN
2002      6,783.0            8,910.0     5,639.0
2003      6,900.0            8,379.0         NaN
2004      7,283.0            8,723.0         NaN
2005      7,117.0            8,695.0         NaN
2006      7,150.0            8,593.0         NaN
2007      7,342.0            8,778.0     6,146.0
2008      7,525.0            8,683.0         NaN
2009      6,933.0            8,038.0         NaN
2010      7,233.0            8,301.0         NaN
2011      7,358.0            8,469.0         NaN
2012      7,308.0            8,339.0     6,864.0
2013      7,158.0            8,136.0         NaN
2014      7,250.0            8,171.0         NaN
2015      7,400.0            8,159.0         NaN
2016      7,692.0            8,430.0         NaN
2017      6,917.0            7,332.0     6,737.0
2018      4,233.0   

---
## 7. QCEW Employment, Compensation, and Payroll

**Source:** BLS Quarterly Census of Employment and Wages (QCEW), USVI, private sector.  
**Coverage:** 2000—2025, annual.  
**Industry scope:** NAICS 72 (Accommodation & food services combined), 721 (Accommodation), 722 (Food services & drinking places).

QCEW measures **covered employment** — jobs covered by unemployment insurance — and **wages paid to those workers**. It excludes self-employment and workers not covered by UI laws. QCEW employment is a job count, not a unique-person count.

`avg_annual_pay` = total wages / average employment. This is the average wage across covered jobs in the industry, not the earnings of a typical individual worker, who may hold multiple jobs or change employers.

Wage comparisons over time use the BLS CPI-U (national average) as a deflator. CPI-U is a mainland U.S. proxy; it is not a USVI-specific price index. Adjusted values are labeled "approximate 2016 dollars (CPI-U proxy)."


In [12]:
qcew = load_qcew(deflate=True, cpi_base_year=2016)
qcew_usvi = qcew.query("area_fips == '78000'")
cpi = load_cpi_annual(base_year=2016)

print("=== QCEW coverage ===")
print(f"Years: {qcew_usvi.year.min()}—{qcew_usvi.year.max()}")
print(f"Industries: {qcew_usvi.naics.unique().tolist()}")
print()
print(
    qcew_usvi[
        qcew_usvi.year.isin([2000, 2010, 2016, 2019, 2022, 2024, 2025])
    ].to_string(index=False)
)

=== QCEW coverage ===
Years: 2000—2025
Industries: ['72', '721', '722']

 year area_fips area_name geography naics                     naics_label disclosure_code  annual_avg_estabs  annual_avg_emplvl  total_annual_wages  annual_avg_wkly_wage  avg_annual_pay  avg_annual_pay_real  total_annual_wages_real  cpi_base_year
 2000     78000      USVI Territory    72   Accommodation & food services                              317.0            6,312.0       106,332,138.0                 324.0        16,845.0             23,478.0            148,202,527.0           2016
 2010     78000      USVI Territory    72   Accommodation & food services                              332.0            6,380.0       151,625,697.0                 457.0        23,766.0             26,159.0            166,889,869.0           2016
 2016     78000      USVI Territory    72   Accommodation & food services                              343.0            6,775.0       183,884,538.0                 522.0        27,142.0 

In [13]:
# --- Annual average employment by industry ---
fig = go.Figure()
for naics, label, color, dash in [
    ("72", "Accommodation & food services", PALETTE["total"], "solid"),
    ("721", "Accommodation", PALETTE["tourists"], "dash"),
    ("722", "Food services", PALETTE["cruise"], "dot"),
]:
    sub = qcew_usvi[qcew_usvi["naics"] == naics]
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub["annual_avg_emplvl"],
            mode="lines",
            name=label,
            line=dict(color=color, width=2.2 if naics == "72" else 1.6, dash=dash),
        )
    )

fig.update_layout(
    title=dict(
        text="<b>QCEW Annual Average Employment — Accommodation & Food Services</b>",
        font=dict(size=16),
        x=0.0,
    ),
    yaxis_title="Covered jobs (annual average)",
    yaxis_tickformat=",",
    height=420,
    annotations=[
        source_note(
            "BLS QCEW, USVI private sector (own_code=5). Covered employment only."
        )
    ],
)
add_shocks(fig)
fig.show()

# Print key years
print("Annual avg employment (covered jobs):")
print(
    qcew_usvi[
        qcew_usvi.year.isin(
            [2000, 2007, 2010, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2024, 2025]
        )
    ][["year", "naics", "naics_label", "annual_avg_emplvl"]]
    .set_index(["year", "naics"])
    .to_string()
)

Annual avg employment (covered jobs):
                                naics_label  annual_avg_emplvl
year naics                                                    
2000 72       Accommodation & food services            6,312.0
2007 72       Accommodation & food services            6,461.0
2010 72       Accommodation & food services            6,380.0
2016 72       Accommodation & food services            6,775.0
2017 72       Accommodation & food services            6,111.0
2018 72       Accommodation & food services            3,619.0
2019 72       Accommodation & food services            4,864.0
2020 72       Accommodation & food services            4,324.0
2021 72       Accommodation & food services            5,125.0
2022 72       Accommodation & food services            5,542.0
2024 72       Accommodation & food services            6,186.0
2025 72       Accommodation & food services            6,076.0
2000 721                      Accommodation            4,227.0
2007 721         

In [14]:
# --- Average annual pay: nominal and CPI-adjusted ---
fig = go.Figure()
for naics, label, color in [
    ("72", "NAICS 72 nominal", PALETTE["total"]),
    ("72", "NAICS 72 real", PALETTE["air_visitors"]),
    ("721", "NAICS 721 nominal", PALETTE["tourists"]),
    ("722", "NAICS 722 nominal", PALETTE["cruise"]),
]:
    sub = qcew_usvi[qcew_usvi["naics"] == naics]
    col = "avg_annual_pay" if "nominal" in label else "avg_annual_pay_real"
    dash = "solid" if "nominal" in label else "dash"
    lw = 2.2 if naics == "72" else 1.5
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub[col],
            mode="lines",
            name=label,
            line=dict(color=color, width=lw, dash=dash),
            opacity=1.0 if naics == "72" else 0.65,
        )
    )

fig.update_layout(
    title=dict(
        text="<b>QCEW Average Annual Pay per Covered Job</b><br>"
        "<span style='font-size:12px;color:#6b7280'>Nominal USD and approximate 2016 dollars (CPI-U proxy). NAICS 72 highlighted.</span>",
        font=dict(size=16),
        x=0.0,
    ),
    yaxis_title="Avg annual pay per covered job ($)",
    yaxis_tickformat="$,.0f",
    height=440,
    annotations=[
        source_note(
            "BLS QCEW, USVI private sector. Real = nominal — (CPI2016 / CPI_year). CPI-U is a mainland proxy."
        )
    ],
)
add_shocks(fig)
fig.show()

In [15]:
# --- Total annual wages (payroll) nominal and real ---
n72 = qcew_usvi[qcew_usvi["naics"] == "72"].copy()

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=n72["year"],
        y=n72["total_annual_wages"] / 1e6,
        name="Nominal",
        marker_color=PALETTE["total"],
        opacity=0.75,
    )
)
fig.add_trace(
    go.Scatter(
        x=n72["year"],
        y=n72["total_annual_wages_real"] / 1e6,
        mode="lines",
        name="Approx. 2016 $ (CPI-U proxy)",
        line=dict(color=PALETTE["air_visitors"], width=2.2),
    )
)
fig.update_layout(
    title=dict(
        text="<b>QCEW Total Annual Wages — NAICS 72 (Nominal and Real)</b>",
        font=dict(size=16),
        x=0.0,
    ),
    yaxis_title="Total annual wages ($M)",
    yaxis_tickformat="$,.0f",
    height=420,
    barmode="overlay",
    annotations=[
        source_note(
            "BLS QCEW, USVI private sector. Bars = nominal. Line = approximate 2016 dollars (CPI-U proxy)."
        )
    ],
)
add_shocks(fig)
fig.show()

print("NAICS 72 total wages ($M) — nominal and approximate 2016 $ (CPI-U proxy):")
display_cols = ["year", "total_annual_wages", "total_annual_wages_real"]
n72_print = n72[display_cols].copy()
n72_print["total_annual_wages"] = (n72_print["total_annual_wages"] / 1e6).round(1)
n72_print["total_annual_wages_real"] = (
    n72_print["total_annual_wages_real"] / 1e6
).round(1)
print(n72_print.to_string(index=False))

NAICS 72 total wages ($M) — nominal and approximate 2016 $ (CPI-U proxy):
 year  total_annual_wages  total_annual_wages_real
 2000               106.3                    148.2
 2001               113.1                    153.2
 2002               113.4                    151.3
 2003               122.9                    160.3
 2004               132.7                    168.7
 2005               137.7                    169.3
 2006               140.8                    167.6
 2007               154.4                    178.8
 2008               161.4                    179.9
 2009               142.4                    159.3
 2010               151.6                    166.9
 2011               154.6                    165.0
 2012               157.5                    164.6
 2013               159.2                    164.1
 2014               164.7                    167.0
 2015               180.6                    182.9
 2016               183.9                    183.9
 2017   

In [16]:
# --- Recovery index (2016 = 100) for employment, pay, payroll ---
BASE = 2016

n72 = qcew_usvi[qcew_usvi["naics"] == "72"].set_index("year")


def _idx(s, base=BASE):
    bv = s[base] if base in s.index else None
    if bv is None or bv == 0:
        return s * np.nan
    return s / bv * 100


emp_idx = _idx(n72["annual_avg_emplvl"])
pay_idx = _idx(n72["avg_annual_pay_real"])
wages_idx = _idx(n72["total_annual_wages_real"])

fig = go.Figure()
for idx_s, label, color, dash in [
    (emp_idx, "Employment", PALETTE["employment"], "solid"),
    (pay_idx, "Real avg pay", PALETTE["air_visitors"], "dash"),
    (wages_idx, "Real total payroll", PALETTE["value_added"], "dot"),
]:
    fig.add_trace(
        go.Scatter(
            x=idx_s.index,
            y=idx_s.values,
            mode="lines",
            name=label,
            line=dict(color=color, width=2.0, dash=dash),
        )
    )

fig.add_hline(
    y=100,
    line=dict(color="#d1d5db", width=1, dash="dot"),
    annotation_text="2016 = 100",
    annotation_position="right",
    annotation_font=dict(size=9, color="#aaa"),
)
fig.update_layout(
    title=dict(
        text="<b>QCEW Recovery Index — NAICS 72 (2016 = 100)</b>",
        font=dict(size=16),
        x=0.0,
    ),
    yaxis_title="Index (2016 = 100)",
    height=420,
    annotations=[source_note("BLS QCEW. Real values deflated by CPI-U (2016 base).")],
)
add_shocks(fig)
fig.show()

print("QCEW NAICS 72 recovery index (2016 = 100):")
idx_df = pd.DataFrame(
    {
        "employment": emp_idx,
        "real_avg_pay": pay_idx.round(1),
        "real_payroll": wages_idx.round(1),
    }
)
print(idx_df.loc[2010:].to_string())

QCEW NAICS 72 recovery index (2016 = 100):
      employment  real_avg_pay  real_payroll
year                                        
2010        94.2          96.4          90.8
2011        96.7          92.8          89.7
2012        94.9          94.4          89.5
2013        92.9          96.0          89.2
2014        94.1          96.5          90.8
2015        97.2         102.4          99.5
2016       100.0         100.0         100.0
2017        90.2         102.5          92.4
2018        53.4          92.6          49.5
2019        71.8          93.0          66.8
2020        63.8          93.3          59.6
2021        75.6          99.9          75.6
2022        81.8         101.8          83.2
2023        88.9          99.8          88.7
2024        91.3         103.1          94.1
2025        89.7         107.3          96.2


---
## 8. Island-Level QCEW Employment (NAICS 721 — Accommodation)

The QCEW file includes island-level observations for St. Croix (78010), St. John (78020), and St. Thomas (78030). These are the most granular available data, but are subject to heavy suppression (disclosure_code = N).

**Coverage note:** NAICS 721 (Accommodation) is the best-covered series. NAICS 72 and 722 are suppressed at the island level for most years. St. Croix and St. John 721 are suppressed for most years after 2020.

Suppress = observations coded N are set to NaN, not zero. Do not interpret gaps as zero employment.


In [17]:
# Island-level QCEW NAICS 721 (accommodation)
# Using load_qcew which already includes island rows
from src.cleaning import load_qcew
qcew_all = load_qcew(deflate=True, cpi_base_year=2016)
isl721 = qcew_all[(qcew_all['area_fips'].isin(['78010','78020','78030'])) &
                   (qcew_all['naics'] == '721')][[
    'area_name','year','disclosure_code','annual_avg_emplvl','avg_annual_pay'
]].sort_values(['area_name','year'])
print('NAICS 721 by island (accommodation):')
print(isl721.to_string())


NAICS 721 by island (accommodation):
      area_name  year disclosure_code  annual_avg_emplvl  avg_annual_pay
103   St. Croix  2000                              731.0        15,853.0
104   St. Croix  2001                              606.0        16,925.0
105   St. Croix  2002                              641.0        16,379.0
106   St. Croix  2003                              648.0        16,720.0
107   St. Croix  2004                              677.0        17,115.0
108   St. Croix  2005                              799.0        19,230.0
109   St. Croix  2006                              821.0        20,289.0
110   St. Croix  2007               N                NaN             NaN
111   St. Croix  2008               N                NaN             NaN
112   St. Croix  2009               N                NaN             NaN
113   St. Croix  2010               N                NaN             NaN
114   St. Croix  2011               N                NaN             NaN
115   St. Croi

In [18]:
# St. Thomas accommodation employment - most complete island series
stt_721 = qcew_all[(qcew_all['area_fips']=='78030') & (qcew_all['naics']=='721')].copy()
fig = go.Figure()
fig.add_trace(go.Scatter(x=stt_721['year'], y=stt_721['annual_avg_emplvl'],
    mode='lines+markers', name='St. Thomas NAICS 721',
    line=dict(color=PALETTE['air_visitors'], width=2.3)))
fig.update_layout(title=dict(
        text=_ch._title_text(
            'St. Thomas Accommodation Employment (NAICS 721)',
            'Only island with continuous QCEW coverage 2000–2025'
        ),
        font=dict(size=16),
        x=0.0,
    ),
    yaxis_title='Annual average covered jobs', height=400,
    annotations=[_ch.source_note('BLS QCEW; UI-covered jobs; St. Thomas (FIPS 78030)')])
_ch.add_shocks(fig)
fig.show()
b16 = stt_721.loc[stt_721['year']==2016, 'annual_avg_emplvl'].values[0]
b24 = stt_721.loc[stt_721['year']==2024, 'annual_avg_emplvl'].values[0]
b25 = stt_721.loc[stt_721['year']==2025, 'annual_avg_emplvl'].values[0]
print(f'St. Thomas accommodation (NAICS 721):')
print(f'  2016: {b16:.0f}  2024: {b24:.0f} ({b24/b16*100:.0f}%)  2025: {b25:.0f} ({b25/b16*100:.0f}%)')


St. Thomas accommodation (NAICS 721):
  2016: 2156  2024: 1912 (89%)  2025: 1980 (92%)


In [19]:
# Suppression summary across island-NAICS combinations
island_rows = qcew_all[qcew_all['area_fips'].isin(['78010','78020','78030'])].copy()
island_rows['suppressed'] = island_rows['annual_avg_emplvl'].isna()
summary = island_rows.groupby(['area_name','naics'])['suppressed'].agg(
    total='count', suppressed_n='sum'
).assign(pct_suppressed=lambda d: (d['suppressed_n']/d['total']*100).round(0))
print("Suppression rates by island and NAICS (2000-2025):")
print(summary.to_string())
print()
print("Note: NAICS 72 is never reported at island level (100% suppressed).")
print("Note: 721 and 722 are suppressed in many years for St. Croix and St. John.")


Suppression rates by island and NAICS (2000-2025):
                  total  suppressed_n  pct_suppressed
area_name  naics                                     
St. Croix  72        25            13            52.0
           721       26            13            50.0
           722       25            17            68.0
St. John   72        25            15            60.0
           721       26            13            50.0
           722       25            19            76.0
St. Thomas 72        25             3            12.0
           721       26             0             0.0
           722       25             3            12.0

Note: NAICS 72 is never reported at island level (100% suppressed).
Note: 721 and 722 are suppressed in many years for St. Croix and St. John.


**Conclusion on island employment:** The island-level QCEW data are too heavily suppressed to support a complete island comparison for all three islands. St. Thomas (NAICS 721) is the only island with continuous coverage and shows accommodation employment reaching 89% of its 2016 level in 2024, rising to 92% in 2025. St. Croix and St. John accommodation employment are suppressed for most of the post-2020 period and cannot be reliably compared without additional disclosure.
